# Lab 5: From Beautiful Soup to a real Scrapy project
**Course:** Information Retrieval (IR 2026), CADT  |  **Week 5**  |  **Duration:** 3 hours  |  **STUDENT VERSION — fill the TODO cells**

In Weeks 1–4, you used a prepared document collection. This week you **collect documents from a public scraping sandbox** and prepare a corpus for information retrieval.

**Our test website:** [Books to Scrape](https://books.toscrape.com/) — intentionally built for web-scraping practice (about 1,000 book records; 20 per listing page). Its book prices and ratings are dummy values, not trustworthy commerce data. **Internet access is required**.

**Part A — Beautiful Soup:** explicitly fetch HTML, inspect robots and encoding, extract cards, follow pagination, and request book detail pages.

**Part B — Scrapy:** use the terminal to create a real project; implement a spider with `parse()` and `parse_book()`, follow links, configure polite crawling, export JSON/CSV, and validate an IR-ready corpus.

**Learning outcomes**
1. Distinguish HTTP requests, parsing, and a crawler's scheduling responsibilities.
2. Check robots/terms before collecting and throttle **every** request.
3. Extract relative links and structured fields using Beautiful Soup and Scrapy CSS selectors.
4. Crawl a bounded 2-page sample including detail-page text, without duplicate records.
5. Export JSON and CSV and inspect corpus completeness for an IR project.

**Lab pacing (180 min):** setup and site/ethics 15 · HTTP & robots 15 · Beautiful Soup 30 · pagination/detail 25 · break 10 · Scrapy setup 20 · spider implementation 40 · QA/export 15 · project checklist 10.

**Crawl boundary:** **2 listing pages maximum** in the core exercises. Do not crawl the entire 1,000-book site for this lab. Keep the student/solution project directories separate.

## Part 0. Environment, target site, and safety (15 min)
**Before class:** confirm Python 3, Jupyter, a terminal, and internet access. Use the same Python environment in both Jupyter and terminal. You may install packages there:

```bash
python -m pip install requests beautifulsoup4 pandas "Scrapy>=2.11,<3"
```

**Site use:** only the public dummy-book catalogue; no logins, personal data, payment pages, or anti-bot bypasses. A `robots.txt` file might be absent: that is not permission to disregard terms or site policies. Respect explicit restrictions and stop if access is denied.

In [3]:
from pathlib import Path
from urllib.parse import urljoin
from urllib import robotparser
import hashlib, json, re, time, sys
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://books.toscrape.com/"
LISTING_LIMIT = 2
DELAY_SECONDS = 0.5
HEADERS = {"User-Agent": "CADT-IR-Lab5/1.0 (educational sandbox crawler)"}
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
print("Python:", sys.version.split()[0], "| target:", BASE)
print("Do not run an unbounded crawl. Target: 2 listing pages.")

Python: 3.14.3 | target: https://books.toscrape.com/
Do not run an unbounded crawl. Target: 2 listing pages.


## Part 1. HTTP inspection (15 min, shared with Part 2)
Run a request and inspect the status code, response Content-Type, final URL, and initial HTML. **Always set a timeout.** A 200 response alone does not mean automated collection is permitted.

In [5]:
# TODO 1.1: fetch BASE with requests.get, HEADERS, timeout=20.
# TODO: print status_code, response.headers.get("Content-Type") and response.url.
response = None
import requests

response = requests.get(BASE, headers=HEADERS, timeout=20)

print("Status Code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Final URL:", response.url)
# Your code here

Status Code: 200
Content-Type: text/html
Final URL: https://books.toscrape.com/


**Answer 1.1:** What do HTTP 200, 403, 404 and 429 mean? What should a responsible crawler do after a 429 response? **Write here:** the HTTP 200 mean that our website have a good respond. And the 403 mean that our website error means that the web server understands your request but refuses to authorize it. And the 404 mean that our url is not found. And 429 it's mean that our website has many request.

## Part 2. Check `robots.txt` before crawling
Inspect `https://books.toscrape.com/robots.txt` **at runtime** (it may return 404). When it has a valid robots file, parse and follow its rules. The demo site's public use does not generalize to your course-project data sources.

In [6]:
robots_url = urljoin(BASE, "robots.txt")
robots_response = requests.get(robots_url, headers=HEADERS, timeout=20)
print("robots.txt HTTP status:", robots_response.status_code)
rp = robotparser.RobotFileParser()
robots_exists = robots_response.status_code == 200
if robots_exists:
    rp.parse(robots_response.text.splitlines())
    print("First 500 characters:", robots_response.text[:500])
    print("Can fetch catalogue?", rp.can_fetch(HEADERS["User-Agent"], BASE))
    print("Declared crawl delay:", rp.crawl_delay(HEADERS["User-Agent"]))
elif robots_response.status_code == 404:
    print("No robots rules published at this path; still review terms and crawl politely.")
else:
    raise RuntimeError("Cannot verify robots.txt safely; stop and investigate.")

def is_allowed(url):
    return (not robots_exists) or rp.can_fetch(HEADERS["User-Agent"], url)

declared_delay = rp.crawl_delay(HEADERS["User-Agent"]) if robots_exists else None
effective_delay = max(DELAY_SECONDS, declared_delay or 0)
print("Request delay in this notebook:", effective_delay, "seconds")

_last_request = [None]
def polite_get(url):
    """Robots-aware, single-threaded request with a delay before *each* next fetch."""
    if not url.startswith(BASE) or not is_allowed(url):
        raise PermissionError(f"Out-of-scope or disallowed URL: {url}")
    if _last_request[0] is not None:
        elapsed = time.monotonic() - _last_request[0]
        time.sleep(max(0.0, effective_delay - elapsed))
    r = requests.get(url, headers=HEADERS, timeout=20)
    _last_request[0] = time.monotonic()
    r.raise_for_status()
    return r

robots.txt HTTP status: 404
No robots rules published at this path; still review terms and crawl politely.
Request delay in this notebook: 0.5 seconds


**Answer 2.1:** Does this site's robots URL publish rules? If it returns 404, is that equivalent to a license to scrape *any* website? How do robots, website terms, and privacy differ? **Write here:** It returns 404. so it mean that there no robots.txt file so we dont have permission to scrape this website.

## Part 3. Beautiful Soup: parse a listing page (30 min)
View the page source and use browser DevTools: each book card is `article.product_pod`; its title is in `h3 a`, price in `p.price_color`, rating in `p.star-rating`. Absolute URLs need `urljoin(page_url, relative_href)`; never concatenate paths naively.

In [12]:
listing_response = polite_get(BASE)
soup = BeautifulSoup(listing_response.content, "html.parser")
cards = soup.select("article.product_pod")
print("Cards on first listing page:", len(cards))
print(cards[0].prettify()[:900] if cards else "No cards found — inspect HTML/response")

Cards on first listing page: 20
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [13]:
from urllib.parse import urljoin

def parse_card(card, page_url):
    """Return {title, url, price_gbp, rating, availability}."""

    # TODO 3.1: Find the h3 link and its title attribute
    link = card.select_one("h3 a")
    title = link.get("title") if link else None

    # TODO 3.2: Resolve the relative URL
    href = link.get("href") if link else None
    url = urljoin(page_url, href) if href else None

    # TODO 3.3: Parse numeric GBP price
    price_element = card.select_one(".price_color")
    price_text = price_element.get_text(strip=True) if price_element else None

    price_gbp = (
        float(price_text.replace("£", "").replace(",", ""))
        if price_text else None
    )

    # TODO 3.4: Map One..Five CSS class to rating integer
    rating_element = card.select_one(".star-rating")
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    rating = None
    if rating_element:
        classes = rating_element.get("class", [])
        for css_class in classes:
            if css_class in rating_map:
                rating = rating_map[css_class]
                break

    # TODO 3.5: Collect availability text
    availability_element = card.select_one("p.instock.availability")
    availability = (
        availability_element.get_text(strip=True)
        if availability_element else None
    )

    return {
        "title": title,
        "url": url,
        "price_gbp": price_gbp,
        "rating": rating,
        "availability": availability
    }

In [14]:
first_page_books = [parse_card(card, listing_response.url) for card in cards]
display(pd.DataFrame(first_page_books).head(5))
assert first_page_books and all(b["url"].startswith(BASE) for b in first_page_books)
print("Unique URLs:", len({b["url"] for b in first_page_books}))

,title,url,price_gbp,rating,availability
0,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,51.77,3,In stock
1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...,53.74,1,In stock
2,Soumission,https://books.toscrape.com/catalogue/soumissio...,50.10,1,In stock
3,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...,47.82,4,In stock
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...,54.23,5,In stock


Unique URLs: 20


### Encoding observation (replaces the old artificial bug)
The original offline site intentionally produced a wrongly decoded `£` symbol. **Do not assume** the same error happens here. Compare the HTTP charset, `requests`' chosen encoding, its apparent encoding and the `£` seen in the HTML. Passing `response.content` to Beautiful Soup allows it to detect bytes/HTML declarations rather than relying on a possibly incorrect `response.text` decoding.

In [15]:
print("HTTP Content-Type:", listing_response.headers.get("Content-Type"))
print("requests encoding:", listing_response.encoding)
print("apparent encoding:", listing_response.apparent_encoding)
print("Price from bytes / Beautiful Soup:", soup.select_one("p.price_color").get_text(strip=True))
print("Price from requests.text:", BeautifulSoup(listing_response.text, "html.parser")
      .select_one("p.price_color").get_text(strip=True))

HTTP Content-Type: text/html
requests encoding: ISO-8859-1
apparent encoding: utf-8
Price from bytes / Beautiful Soup: £51.77
Price from requests.text: Â£51.77


## Part 4. Beautiful Soup: pagination and book-detail text (25 min)
Two types of URLs: `li.next a` finds the *next listing page*; `h3 a` finds each book's *detail page*. Keep a visited-URL set; cap **listing pages at two**. Request details for only **three books** here (Scrapy collects more later). `polite_get()` throttles each request.

**Answer 3.1:** Did you observe mojibake, or were both price strings correct? Why must a parser handle missing fields without crashing on an unfamiliar website? **Write here:**  I observe mojibake becuase the price from requests.text is Â£51.77 , instead of £51.77 this happend becuase the request used  ISO-8859-1 and the apparent use utf-8. A perser should handle missing fileds without crashing because diffrent website have many diffrent body and HTML structture.

In [26]:
def collect_listing_books(start_url, max_pages=2):
    """Return (books, listing_pages_visited)."""

    books = []
    visited = set()
    current_url = start_url

    while current_url and len(visited) < max_pages:
        if current_url in visited:
            break

        visited.add(current_url)

        response = polite_get(current_url)
        soup = BeautifulSoup(response.content, "html.parser")

        cards = soup.select("article.product_pod")

        for card in cards:
            books.append(parse_card(card, response.url))

        next_link = soup.select_one("li.next a")

        if next_link and next_link.get("href"):
            current_url = urljoin(response.url, next_link["href"])
        else:
            current_url = None

    return books, list(visited)

In [27]:
listing_books, pages_visited = collect_listing_books(BASE, max_pages=LISTING_LIMIT)

print("Listing pages visited:", len(pages_visited), "| listing records:", len(listing_books))

assert 1 <= len(pages_visited) <= LISTING_LIMIT
assert len({b["url"] for b in listing_books}) == len(listing_books)

display(pd.DataFrame(listing_books).head(3))

Listing pages visited: 2 | listing records: 40


,title,url,price_gbp,rating,availability
0,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,51.77,3,In stock
1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...,53.74,1,In stock
2,Soumission,https://books.toscrape.com/catalogue/soumissio...,50.10,1,In stock


In [24]:
def parse_book_detail(url):
    """Return complete record with title, description, doc_id, and IR text."""

    response = polite_get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    title_element = soup.select_one("div.product_main h1")
    title = title_element.get_text(strip=True) if title_element else None

    description_element = soup.select_one("#product_description + p")
    description = description_element.get_text(strip=True) if description_element else None

    category_element = soup.select_one("ul.breadcrumb li:nth-child(3) a")
    category = category_element.get_text(strip=True) if category_element else None

    doc_id = url.rstrip("/").split("/")[-1]

    text = " ".join(filter(None, [title, description, category]))

    return {
        "doc_id": doc_id,
        "title": title,
        "text": text,
        "description": description,
        "url": response.url,
        "category": category,
        "source": "Books to Scrape"
    }

In [25]:
bs_details = [parse_book_detail(book["url"]) for book in listing_books[:3]]
display(pd.DataFrame(bs_details)[["doc_id", "title", "category", "description"]])
Path("books_bs_sample.json").write_text(
    json.dumps(bs_details, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved 3 Beautiful Soup detail records to books_bs_sample.json")

,doc_id,title,category,description
0,index.html,A Light in the Attic,Poetry,It's hard to imagine a world without A Light i...
1,index.html,Tipping the Velvet,Historical Fiction,"""Erotic and absorbing...Written with starling ..."
2,index.html,Soumission,Fiction,"Dans une France assez proche de la nôtre, un h..."


Saved 3 Beautiful Soup detail records to books_bs_sample.json


**Answer 4.1:** What would happen if you used `BASE + href` instead of `urljoin(current_page, href)`? Why do we add `title` to `description` in the `text` field for IR? **Write here:**  when i use `Base + href` instead of  `urljoin(current_page, href)` it will produce incorrect URL. We need to add `title` in the `text` because we need the description  in the `text` fields for information retrieval.

## Break (10 min)

## Part 5. Introduce Scrapy — a real crawler project (20 min)
Beautiful Soup only **parses**; Requests **downloads**. Scrapy additionally provides a request scheduler, asynchronous crawling, built-in robots handling, retry controls, CSS selectors, and feed exporters.

**In a terminal** (not a Python cell), go to the directory containing this notebook and run:

```bash
scrapy version
scrapy startproject bookcrawler
cd bookcrawler
scrapy genspider books books.toscrape.com
scrapy list
```

Do this yourselves: **building the project is part of the assessment**. If your group cannot scaffold during class, a `Student_Starter` project is included as a recovery option; it must still be implemented by you.

**Project tree:**
```text
bookcrawler/                      # terminal working directory
├── scrapy.cfg
└── bookcrawler/
    ├── settings.py               # configure politeness
    └── spiders/
        └── books.py              # YOUR spider implementation
```

In `bookcrawler/settings.py`, configure:

```python
ROBOTSTXT_OBEY = True
CONCURRENT_REQUESTS_PER_DOMAIN = 1
DOWNLOAD_DELAY = 0.5
DOWNLOAD_TIMEOUT = 20
USER_AGENT = "CADT-IR-Lab5/1.0 (educational sandbox crawler)"
FEED_EXPORT_ENCODING = "utf-8"
```

**Check site access:** `scrapy shell "https://books.toscrape.com/"`. If `robots.txt` declares a longer crawl delay than 0.5 seconds, increase `DOWNLOAD_DELAY` accordingly.

Inside Scrapy shell, practice:

```python
len(response.css("article.product_pod"))
response.css("article.product_pod h3 a::attr(href)").get()
response.css("li.next a::attr(href)").get()
```

Exit with `exit()` and test a **book detail page** using `scrapy shell "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"`.
Try: `response.css("div.product_main h1::text").get()` and `response.css("#product_description + p ::text, #product_description + p::text").getall()`.

**Answer 5.1:** What does `yield response.follow(href, callback=self.parse_book)` do that `return href` does not? **Write here:**

## Part 6. Implement a Scrapy spider (40 min)
Open `bookcrawler/bookcrawler/spiders/books.py` in your editor. Scrapy ships a generated class; replace it with a spider containing:

- `name = "books"`, `allowed_domains`, `start_urls`.
- `__init__` with `max_pages` (default 2) and `pages_seen = 0`.
- `parse(response)` to select listing cards, `yield response.follow()` to detail pages, and follow `li.next a` until two listing pages are processed.
- `parse_book(response)` to yield **one dict per detail page**: `doc_id`, `title`, `text`, `description`, `url`, `category`, `price_gbp`, `rating`, `availability`, `source`.

**Compare responsibilities:**

| Beautiful Soup | Scrapy |
|---|---|
| `soup.select('article.product_pod')` | `response.css('article.product_pod')` |
| `tag.get('href')` | `card.css('a::attr(href)').get()` |
| `urljoin(page_url, href)` | `response.follow(href, callback=...)` |
| `requests.get` + `time.sleep` | Engine + `DOWNLOAD_DELAY` |
| `json.dump(records, ...)` | `scrapy crawl books -O books_scrapy.json` |

Remember: `.get()` returns the first match or `None`; `.getall()` returns all matches. Scrapy returns text/attribute values from `::text` / `::attr(...)` selectors.

**Student task:** write the spider yourself or implement the TODOs in the provided starter. Avoid parsing the page with Beautiful Soup inside your Scrapy spider; practice the Scrapy selector API.

**Spider skeleton (complete each TODO in your project):**

    ```python
    import hashlib, re, scrapy

    RATINGS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    class BooksSpider(scrapy.Spider):
        name = "books"
        allowed_domains = ["books.toscrape.com"]
        start_urls = ["https://books.toscrape.com/"]

        def __init__(self, max_pages=2, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.max_pages = max(1, int(max_pages))
            self.pages_seen = 0

        def parse(self, response):
            self.pages_seen += 1
            # TODO: for each card, yield response.follow(href, callback=self.parse_book)
            # TODO: if pages_seen < max_pages, follow li.next a

        def parse_book(self, response):
            # TODO: extract full detail fields, clean missing values
            # TODO: yield an IR-ready dict (see field list above)
            pass
    ```

    **Tip:** reuse the `RATING_MAP` idea and `re.search()` from Part 3. Use `hashlib.sha1(response.url.encode('utf-8')).hexdigest()[:12]` for a stable, URL-derived document ID.

### Run a small crawl and export (use the terminal inside `bookcrawler/`)

```bash
# Debug one listing page (~20 books, each with a detail request)
scrapy crawl books -a max_pages=1 -O books_scrapy_preview.json

# Main bounded crawl (~40 details from 2 listing pages); export twice in ONE run
scrapy crawl books -a max_pages=2 -O books_scrapy.json -O books_scrapy.csv
```

**Do not disable `ROBOTSTXT_OBEY`.** Inspect Scrapy's log for status/robots problems, errors, items scraped, and requested pages. `-O` overwrites existing outputs. If your source layout changed, fix selectors rather than falsifying data.

**Answer 6.1:** Why are only ~40 book documents expected even though the website has ~1,000? Why might observed count be lower? **Write here:**

## Part 7. Validate exports and make an IR corpus (15 min)
Reopen this notebook. The JSON file must be under `bookcrawler/books_scrapy.json` when you followed the terminal steps above. If you ran the terminal elsewhere, update `scrapy_json_path`. Check duplicates, empty text, schema, and compare at least one book against the Beautiful Soup results.

In [ ]:
scrapy_json_path = Path("bookcrawler") / "books_scrapy.json"
# If notebook was opened from within the Scrapy project folder:
if not scrapy_json_path.exists() and Path("books_scrapy.json").exists():
    scrapy_json_path = Path("books_scrapy.json")
print("Looking for:", scrapy_json_path.resolve())
if not scrapy_json_path.exists():
    print("No Scrapy export yet: complete Part 6 in your terminal first.")
else:
    scrapy_df = pd.read_json(scrapy_json_path)
    print("Rows:", len(scrapy_df), "| columns:", list(scrapy_df.columns))
    display(scrapy_df.head(3))

In [ ]:
# TODO 7.1: work after books_scrapy.json has been created.
# 1) check uniqueness of `doc_id` and `url`.
# 2) report empty `title`, `description`, `text` counts.
# 3) compare a shared URL with Beautiful Soup's title.
# 4) write only `doc_id`, `title`, `text`, `url`, `category`, `source`
#    into an index-ready JSONL file called `ir_books_corpus.jsonl`.
# Hint: json.dumps(..., ensure_ascii=False) + "\n".

**Answer 7.1:** What is the difference between an HTML page, a scraped structured item, and an IR document? Why is an empty description not always a crawler failure? **Write here:**

## Part 8. Your real project: source-and-ethics checklist (10 min)
**Do not scrape your chosen project website in this lab.** Before your team collects real project data, prepare answers for instructor review:

1. **Source** — website(s) or API(s), and why the documents support an IR search task.
2. **Permission and reuse** — official API availability, site terms, data licenses, and attribution/redistribution scope.
3. **Robots and access** — robots status, disallowed paths, authorization, authentication and crawl restrictions.
4. **Bounded crawl** — page/item target, depth, domain scope, delay/concurrency, stop conditions, error/429 behavior.
5. **Document schema** — stable `doc_id`, `title`, `text`, `url`, `source`, category/date if available; how to handle missing data and duplicates.
6. **Personal data** — whether documents contain identifiable information; collection minimization, security, and retention.
7. **Quality test** — how you will manually verify at least 5–10 records before scaling to project size.

**Your team's answers:**

*(Write here; obtain instructor sign-off before starting a real crawl.)*

## Part 9. Reflection and submission
**Reflection (4–6 sentences):** Compare manual Beautiful Soup pagination with Scrapy's `response.follow` workflow. Name one benefit of a Scrapy project for your IR corpus, one practical failure to guard against, and one restriction you will respect when collecting project data.

---
### Submission checklist
- [ ] All `# TODO` cells completed and the notebook runs from top to bottom without errors
- [ ] All *Answer* cells written in your own words
- [ ] scrapy project (as .zip or GitHub URL)
- [ ] Part 8 ethics checklist completed for your OWN project source(s)
- [ ] Notebook saved as `Lab5_<StudentID>.ipynb` and submitted as instructed

**References:** [Scrapy tutorial](https://docs.scrapy.org/en/latest/intro/tutorial.html) · [Scrapy commands](https://docs.scrapy.org/en/latest/topics/commands.html) · [Scrapy settings](https://docs.scrapy.org/en/latest/topics/settings.html) · [Books to Scrape](https://books.toscrape.com/).